# medi-LLaMA — DPO Fine-Tuning (Person 3)

**Goal:** Take the best SFT checkpoint (Trial 5) and apply DPO (Direct Preference Optimisation) using the MedCRAFT preference pairs. Run 5 trials, pick the best, and produce the final base → SFT → DPO comparison.

**Runtime:** Google Colab T4 GPU (~20–35 min per trial)

**Files needed (upload in Cell 3):**
- `best_checkpoint.zip` — contains `trial_5/` (the SFT adapter)
- `Base-Model.zip` — contains `dpo/` dataset and `test_prompts.json`

## Cell 1 — Install packages

In [ ]:
!pip install -q transformers==4.40.2 peft==0.10.0 trl==0.8.6 \
             datasets accelerate bitsandbytes sacrebleu bert_score

## Cell 2 — Imports & config

In [ ]:
import os, json, time, gc
import torch
import pandas as pd
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from trl import DPOTrainer, DPOConfig
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score_fn

BASE_MODEL        = 'TinyLlama/TinyLlama_v1.1'
SFT_ADAPTER_DIR   = '/content/trial_5'
DPO_DATA_DIR      = '/content/Base-Model/dpo'
TEST_PROMPTS_PATH = '/content/Base-Model/test_prompts.json'
OUTPUT_DIR        = '/content/dpo_outputs'

SYSTEM_PROMPT = (
    'You are an experienced and knowledgeable medical professional. '
    'Provide clear, factual, and helpful medical information.'
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('✅ Config ready | device:', device)
if device == 'cpu':
    print('⚠️  No GPU detected! Switch runtime → T4 GPU in Colab before continuing.')

## Cell 3 — Upload & extract files

Run this, then click **Choose Files** and select both zip files.

In [ ]:
from google.colab import files
import zipfile

print('Upload best_checkpoint.zip and Base-Model.zip')
uploaded = files.upload()

for fname in uploaded:
    print(f'Extracting {fname}...')
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content')
    print(f'  ✅ Done')

# Verify
checks = [
    ('/content/trial_5/adapter_config.json', 'SFT adapter'),
    ('/content/Base-Model/dpo/train',         'DPO train data'),
    ('/content/Base-Model/test_prompts.json', 'Test prompts'),
]
for path, label in checks:
    status = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'  {status}  {label}')

## Cell 4 — Load tokenizer & test prompts

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

with open(TEST_PROMPTS_PATH) as f:
    TEST_PROMPTS = json.load(f)
REFERENCES = [p['gold_answer'] for p in TEST_PROMPTS]

print(f'✅ Tokenizer ready | {len(TEST_PROMPTS)} test prompts loaded')
print('\nPrompt categories:', [p['category'] for p in TEST_PROMPTS])

## Cell 5 — Load & format DPO dataset

The MedCRAFT DPO split uses Alpaca `### Instruction:` format. We reformat it to match TinyLlama's chat template — the same format used during SFT.

In [ ]:
raw_dpo = load_from_disk(DPO_DATA_DIR)
print('DPO dataset:', raw_dpo)
print('Columns:',     raw_dpo['train'].column_names)

def format_dpo_row(row):
    # MedCRAFT prompt: '{system}\n\n### Instruction:\n{question}'
    # → extract just the question, re-wrap in TinyLlama chat template
    raw_prompt = row['prompt']
    if '### Instruction:\n' in raw_prompt:
        instruction = raw_prompt.split('### Instruction:\n', 1)[1].strip()
    else:
        instruction = raw_prompt.strip()

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': instruction},
    ]
    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return {
        'prompt':   formatted_prompt,
        'chosen':   row['chosen'].strip(),
        'rejected': row['rejected'].strip(),
    }

# 3000 train / 300 val — keeps each trial ~20-30 min on T4
n_train = min(3000, len(raw_dpo['train']))
n_val   = min(300,  len(raw_dpo['test']))

dpo_train = raw_dpo['train'].select(range(n_train)).map(
    format_dpo_row, remove_columns=raw_dpo['train'].column_names
)
dpo_val = raw_dpo['test'].select(range(n_val)).map(
    format_dpo_row, remove_columns=raw_dpo['test'].column_names
)

print(f'\n✅ DPO train: {len(dpo_train)} | val: {len(dpo_val)}')
print('\nSample prompt (first 250 chars):')
print(dpo_train[0]['prompt'][:250])
print('\nSample chosen (first 150 chars):')
print(dpo_train[0]['chosen'][:150])

## Cell 6 — Inference & evaluation helper

Same greedy decoding settings used by Person 2's SFT evaluation.

In [ ]:
def generate_response(model, tokenizer, question, max_new_tokens=300):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def evaluate_model(model, tokenizer, label=''):
    print(f'  ⏳ Evaluating [{label}]...')
    preds = [generate_response(model, tokenizer, p['question']) for p in TEST_PROMPTS]

    bleu_metric = BLEU(effective_order=True)
    bleu_scores = [bleu_metric.sentence_score(p, [r]).score
                   for p, r in zip(preds, REFERENCES)]
    avg_bleu = sum(bleu_scores) / len(bleu_scores)

    _, _, F1 = bert_score_fn(preds, REFERENCES, lang='en', verbose=False)
    avg_bert = F1.mean().item()

    print(f'  ✅ BLEU: {avg_bleu:.4f}  |  BERTScore F1: {avg_bert:.4f}')
    return {
        'label': label, 'bleu': round(avg_bleu, 4),
        'bertscore_f1': round(avg_bert, 4), 'predictions': preds
    }

print('✅ Helpers defined')

## Cell 7 — Evaluate base model + SFT Trial 5

This confirms your setup is working before DPO training starts. Scores should match: base BLEU ~1.12, SFT BLEU ~3.34.

In [ ]:
# ── Load base model ─────────────────────────────────────────────────
print('Loading base TinyLlama (~2 min on first run, cached after)...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16,
    device_map='auto', trust_remote_code=True,
)
base_model.eval()
base_results = evaluate_model(base_model, tokenizer, label='Base TinyLlama')

# ── Load SFT adapter on top ───────────────────────────────────────────
print('\nLoading SFT Trial 5 adapter...')
sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_DIR)
sft_model.eval()
sft_results = evaluate_model(sft_model, tokenizer, label='SFT Trial 5')

# ── Quick comparison ─────────────────────────────────────────────────
print('\n' + '='*60)
print('QUICK CHECK — Prompt 1 (Type 2 Diabetes)')
print('='*60)
print(f'  Base:  {base_results["predictions"][0][:250]}...')
print(f'\n  SFT:   {sft_results["predictions"][0][:250]}...')

## Cell 8 — DPO trial configurations

**5 trials, varying 4 hyperparameters:**

| Trial | beta | LR | Batch | Epochs | What we're testing |
|-------|------|----|-------|--------|--------------------|
| 1 | 0.10 | 5e-5 | 2 | 1 | Baseline DPO config |
| 2 | 0.20 | 5e-5 | 2 | 1 | Higher beta (more conservative) |
| 3 | 0.10 | 1e-4 | 2 | 1 | Higher LR with same beta |
| 4 | 0.30 | 5e-5 | 1 | 2 | High beta + more epochs |
| 5 | 0.05 | 3e-5 | 2 | 1 | Low beta (more aggressive update) |

**beta** is the key DPO parameter: lower = bigger shift from SFT, higher = stays close to SFT.

In [ ]:
DPO_TRIALS = [
    {'trial_id': 1, 'beta': 0.10, 'lr': 5e-5,  'batch_size': 2, 'epochs': 1},
    {'trial_id': 2, 'beta': 0.20, 'lr': 5e-5,  'batch_size': 2, 'epochs': 1},
    {'trial_id': 3, 'beta': 0.10, 'lr': 1e-4,  'batch_size': 2, 'epochs': 1},
    {'trial_id': 4, 'beta': 0.30, 'lr': 5e-5,  'batch_size': 1, 'epochs': 2},
    {'trial_id': 5, 'beta': 0.05, 'lr': 3e-5,  'batch_size': 2, 'epochs': 1},
]
print(f'✅ {len(DPO_TRIALS)} DPO trials defined')

## Cell 9 — DPO trial runner function

In [ ]:
def run_dpo_trial(cfg):
    tid = cfg['trial_id']
    trial_dir = f'{OUTPUT_DIR}/dpo_trial_{tid}'
    print(f"\n{'='*65}")
    print(f' DPO TRIAL {tid}  |  beta={cfg["beta"]}  lr={cfg["lr"]}  '
          f'batch={cfg["batch_size"]}  epochs={cfg["epochs"]}')
    print(f"{'='*65}")

    # 1. Load base + merge SFT adapter
    print('  Loading base + merging SFT adapter...')
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16,
        device_map='auto', trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(model, SFT_ADAPTER_DIR)
    model = model.merge_and_unload()

    # 2. Add fresh LoRA for DPO
    dpo_lora = LoraConfig(
        r=32, lora_alpha=64,
        target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
        lora_dropout=0.05, bias='none',
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, dpo_lora)
    model.print_trainable_parameters()

    # 3. Reference model: frozen copy of merged SFT (no LoRA)
    print('  Loading reference model (frozen SFT)...')
    ref_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16,
        device_map='auto', trust_remote_code=True,
    )
    ref_model = PeftModel.from_pretrained(ref_base, SFT_ADAPTER_DIR)
    ref_model = ref_model.merge_and_unload()
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad = False

    # 4. DPO training args
    dpo_args = DPOConfig(
        output_dir=trial_dir,
        num_train_epochs=cfg['epochs'],
        per_device_train_batch_size=cfg['batch_size'],
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=cfg['lr'],
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        bf16=True, fp16=False,
        beta=cfg['beta'],
        max_length=512,
        max_prompt_length=256,
        logging_steps=50,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        report_to='none',
    )

    trainer = DPOTrainer(
        model=model, ref_model=ref_model,
        args=dpo_args,
        train_dataset=dpo_train, eval_dataset=dpo_val,
        tokenizer=tokenizer,
    )

    # 5. Train
    t0 = time.time()
    trainer.train()
    train_time = round((time.time() - t0) / 60, 1)

    val_logs = [l for l in trainer.state.log_history if 'eval_loss' in l]
    val_loss = round(val_logs[-1]['eval_loss'], 4) if val_logs else None

    # 6. Evaluate on 10 test prompts
    trainer.save_model(trial_dir)
    eval_res = evaluate_model(model, tokenizer, label=f'DPO Trial {tid}')

    # 7. Cleanup GPU memory before next trial
    del model, ref_model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return {
        'trial_id': tid, 'config': cfg,
        'val_loss': val_loss, 'train_time_min': train_time,
        'bleu': eval_res['bleu'],
        'bertscore_f1': eval_res['bertscore_f1'],
        'predictions': eval_res['predictions'],
        'checkpoint_dir': trial_dir,
    }

print('✅ run_dpo_trial() defined')

## Cell 10 — Run all 5 DPO trials

⏰ **Expect ~20–35 min per trial on a T4 GPU.** All 5 trials ≈ 2–3 hours total.

In [ ]:
dpo_all_results = []

for cfg in DPO_TRIALS:
    result = run_dpo_trial(cfg)
    dpo_all_results.append(result)
    print(f"  → Trial {result['trial_id']} complete: "
          f"BLEU {result['bleu']}  BERTScore {result['bertscore_f1']}  "
          f"val_loss {result['val_loss']}  ({result['train_time_min']} min)")

print('\n✅ All DPO trials complete!')

## Cell 11 — Select best DPO model

Selection criterion: highest BERTScore F1 → then BLEU → then lowest val loss as tiebreaker.

In [ ]:
# Sort: BERTScore desc → BLEU desc → val_loss asc
ranked = sorted(
    dpo_all_results,
    key=lambda x: (x['bertscore_f1'], x['bleu'], -(x['val_loss'] or 99)),
    reverse=True,
)
best_dpo = ranked[0]

print('\n' + '='*65)
print(f"  BEST DPO MODEL → Trial {best_dpo['trial_id']}")
print(f"  beta={best_dpo['config']['beta']}  lr={best_dpo['config']['lr']}  "
      f"batch={best_dpo['config']['batch_size']}  epochs={best_dpo['config']['epochs']}")
print(f"  BLEU {best_dpo['bleu']}  BERTScore {best_dpo['bertscore_f1']}  "
      f"val_loss {best_dpo['val_loss']}")
print('='*65)
BEST_DPO_CHECKPOINT = best_dpo['checkpoint_dir']

# Rank all DPO trials for report
print('\nAll DPO trials ranked:')
for i, r in enumerate(ranked):
    marker = ' ← BEST' if i == 0 else ''
    print(f"  {i+1}. Trial {r['trial_id']} | BLEU {r['bleu']} | "
          f"BERTScore {r['bertscore_f1']} | val_loss {r['val_loss']}{marker}")

## Cell 12 — Final comparison table (base → SFT → DPO)

In [ ]:
# Numbers from Person 1 and Person 2
BASELINE_BLEU  = 1.1166
BASELINE_BERT  = 0.7788
SFT_BEST_BLEU  = 3.3385
SFT_BEST_BERT  = 0.8292

rows = [
    {'Model': 'Base TinyLlama',   'BLEU': BASELINE_BLEU, 'BERTScore F1': BASELINE_BERT, 'Val Loss': '-', 'Notes': 'No fine-tuning'},
    {'Model': 'SFT Trial 5',      'BLEU': SFT_BEST_BLEU, 'BERTScore F1': SFT_BEST_BERT, 'Val Loss': 1.330, 'Notes': 'rank=64, lr=5e-5'},
]
for r in dpo_all_results:
    best_mark = ' ★' if r['trial_id'] == best_dpo['trial_id'] else ''
    rows.append({
        'Model': f"DPO Trial {r['trial_id']}{best_mark}",
        'BLEU': r['bleu'],
        'BERTScore F1': r['bertscore_f1'],
        'Val Loss': r['val_loss'],
        'Notes': f"beta={r['config']['beta']} lr={r['config']['lr']}"
    })

df_final = pd.DataFrame(rows)
print(df_final.to_string(index=False))

## Cell 13 — Qualitative output comparison

Pick 3 examples showing progression from base → SFT → best DPO.

In [ ]:
SAMPLE_IDS = [0, 5, 7]  # Diabetes, Cardiology, Sepsis

for i in SAMPLE_IDS:
    q  = TEST_PROMPTS[i]['question']
    g  = REFERENCES[i]
    bp = base_results['predictions'][i]
    sp = sft_results['predictions'][i]
    dp = best_dpo['predictions'][i]
    print(f"\n{'─'*65}")
    print(f"Prompt {i+1} [{TEST_PROMPTS[i]['category']}]: {q}")
    print(f"\n  Gold:      {g[:350]}...")
    print(f"\n  Base:      {bp[:350]}...")
    print(f"\n  SFT T5:    {sp[:350]}...")
    print(f"\n  DPO T{best_dpo['trial_id']}: {dp[:350]}...")

## Cell 14 — Save results & download

In [ ]:
final_output = {
    'baseline':  {'bleu': BASELINE_BLEU, 'bertscore_f1': BASELINE_BERT},
    'sft_best':  {'trial_id': 5, 'bleu': SFT_BEST_BLEU, 'bertscore_f1': SFT_BEST_BERT},
    'dpo_trials': [
        {'trial_id': r['trial_id'], 'config': r['config'],
         'bleu': r['bleu'], 'bertscore_f1': r['bertscore_f1'],
         'val_loss': r['val_loss'], 'train_time_min': r['train_time_min']}
        for r in dpo_all_results
    ],
    'best_dpo_trial_id': best_dpo['trial_id'],
    'best_dpo_checkpoint': BEST_DPO_CHECKPOINT,
}

with open('/content/dpo_final_results.json', 'w') as f:
    json.dump(final_output, f, indent=2)
df_final.to_csv('/content/dpo_results_summary.csv', index=False)

print('✅ Saved:')
print('  /content/dpo_final_results.json   ← for report')
print('  /content/dpo_results_summary.csv  ← for report table')
print(f'  {BEST_DPO_CHECKPOINT}/  ← best DPO checkpoint')

# Download results
from google.colab import files
files.download('/content/dpo_final_results.json')
files.download('/content/dpo_results_summary.csv')